# Step 4: Anomaly Detection EDA

This notebook explores the dataset to construct a rolling baseline of events and tunes the Isolation Forest anomaly detector. The goal is to detect sudden spikes in events along specific corridors.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from datetime import timedelta

# Adjust path to actual data location
data_path = '../../data/Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv'
df = pd.read_csv(data_path, low_memory=False)
df['start_datetime'] = pd.to_datetime(df['start_datetime'], errors='coerce')
df = df.dropna(subset=['start_datetime'])
df = df.sort_values('start_datetime')

print(f"Total events loaded: {len(df)}")

## 1. Create Baseline

We use the first 90 days as the baseline to learn the normal distribution of events per hour per corridor.

In [ ]:
min_date = df['start_datetime'].min()
baseline_cutoff = min_date + timedelta(days=90)

baseline_df = df[df['start_datetime'] < baseline_cutoff].copy()
stream_df = df[df['start_datetime'] >= baseline_cutoff].copy()

baseline_df['hour'] = baseline_df['start_datetime'].dt.hour
baseline_df['corridor'] = baseline_df['corridor'].fillna('Non-corridor')

baseline_counts = baseline_df.groupby(['corridor', 'hour']).size().reset_index(name='event_count')
baseline_counts.head()

## 2. Train Isolation Forest

We train the model on `event_count`.

In [ ]:
model = IsolationForest(contamination=0.05, random_state=42)
X_train = baseline_counts[['event_count']].fillna(0)
model.fit(X_train)

baseline_counts['anomaly_score'] = model.predict(X_train)
print("Anomalies in Baseline Training (should be ~5% due to contamination):")
print(baseline_counts['anomaly_score'].value_counts())

## 3. Test on Streaming Data

We simulate a single day of data to detect anomalies.

In [ ]:
stream_df['date'] = stream_df['start_datetime'].dt.date
stream_df['hour'] = stream_df['start_datetime'].dt.hour
stream_df['corridor'] = stream_df['corridor'].fillna('Non-corridor')

# Pick the first date in the stream
test_date = stream_df['date'].iloc[0]
daily_events = stream_df[stream_df['date'] == test_date]

daily_counts = daily_events.groupby(['corridor', 'hour']).size().reset_index(name='event_count')
if not daily_counts.empty:
    X_test = daily_counts[['event_count']].fillna(0)
    daily_counts['is_anomaly'] = model.predict(X_test) == -1
    
    anomalies = daily_counts[daily_counts['is_anomaly'] == True]
    print(f"Detected {len(anomalies)} anomalies for {test_date}")
    display(anomalies)